# Titanic Survival Prediction — Machine Learning Internship Project

**Goal:** Predict whether a passenger survived the Titanic disaster using passenger attributes (class, sex, age, fare, etc.)

**Topics covered:** Python Basics, EDA, Visualization (Matplotlib/Seaborn), Feature Engineering, Statistics, Decision Tree, Random Forest, Model Evaluation & Comparison.

**Dataset:** Titanic passenger dataset (`titanic.csv`) — 891 passengers, 9 columns.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              roc_curve, auc)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (8,5)

pd.set_option('display.max_columns', None)

## 2. Load the Dataset

In [ ]:
df = pd.read_csv("titanic.csv")
print("Shape of dataset:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Exploratory Data Analysis (EDA)

We explore the dataset to understand distributions, missing values, and relationships between features and survival.

### 3.1 Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0]

In [ ]:
plt.figure(figsize=(8,4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title("Missing Values Heatmap")
plt.show()

### 3.2 Target Variable Distribution

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Survived', data=df, palette='Set2')
plt.title("Survival Count (0 = Died, 1 = Survived)")
plt.xlabel("Survived")
plt.ylabel("Number of Passengers")
plt.show()

print(df['Survived'].value_counts(normalize=True).round(3) * 100, "% of total")

### 3.3 Survival by Sex and Passenger Class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))

sns.countplot(x='Sex', hue='Survived', data=df, palette='Set1', ax=axes[0], legend=True)
axes[0].set_title("Survival Count by Sex")

sns.countplot(x='Pclass', hue='Survived', data=df, palette='Set1', ax=axes[1])
axes[1].set_title("Survival Count by Passenger Class")

plt.tight_layout()
plt.show()

### 3.4 Age Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Age'].dropna(), bins=30, kde=True, color='teal')
plt.title("Age Distribution of Passengers")
plt.xlabel("Age")
plt.show()

### 3.5 Fare Distribution by Survival

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='Survived', y='Fare', data=df, palette='Set3')
plt.title("Fare Distribution by Survival")
plt.show()

### 3.6 Correlation Heatmap

In [ ]:
plt.figure(figsize=(8,6))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Heatmap of Numeric Features")
plt.show()

## 4. Basic Statistics

Quick statistical summary to support EDA findings.

In [ ]:
print("Mean Age:", df['Age'].mean().round(2))
print("Median Age:", df['Age'].median())
print("Std Dev of Fare:", df['Fare'].std().round(2))
print()
print("Survival rate by Sex:")
print(df.groupby('Sex')['Survived'].mean().round(3))
print()
print("Survival rate by Pclass:")
print(df.groupby('Pclass')['Survived'].mean().round(3))

## 5. Feature Engineering

- Fill missing `Age` with median (grouped by Pclass for better accuracy)
- Fill missing `Embarked` with mode
- Create `FamilySize` from `SibSp` + `Parch`
- Create `IsAlone` flag
- Encode categorical variables (`Sex`, `Embarked`)
- Bin `Age` into groups

In [ ]:
df_fe = df.copy()

# Fill missing Age using median per Pclass
df_fe['Age'] = df_fe.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))

# Fill missing Embarked with mode
df_fe['Embarked'] = df_fe['Embarked'].fillna(df_fe['Embarked'].mode()[0])

# New feature: Family size
df_fe['FamilySize'] = df_fe['SibSp'] + df_fe['Parch'] + 1

# New feature: IsAlone
df_fe['IsAlone'] = (df_fe['FamilySize'] == 1).astype(int)

# Age bins
df_fe['AgeGroup'] = pd.cut(df_fe['Age'], bins=[0,12,18,35,60,80],
                           labels=['Child','Teen','Adult','MiddleAge','Senior'])

# Encode categoricals
le_sex = LabelEncoder()
df_fe['Sex_enc'] = le_sex.fit_transform(df_fe['Sex'])

le_emb = LabelEncoder()
df_fe['Embarked_enc'] = le_emb.fit_transform(df_fe['Embarked'])

le_age = LabelEncoder()
df_fe['AgeGroup_enc'] = le_age.fit_transform(df_fe['AgeGroup'].astype(str))

df_fe[['Sex','Sex_enc','Embarked','Embarked_enc','AgeGroup','AgeGroup_enc','FamilySize','IsAlone']].head()

### 5.1 Survival Rate by New Features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))

sns.barplot(x='FamilySize', y='Survived', data=df_fe, palette='viridis', ax=axes[0])
axes[0].set_title("Survival Rate by Family Size")

sns.barplot(x='IsAlone', y='Survived', data=df_fe, palette='magma', ax=axes[1])
axes[1].set_title("Survival Rate: Alone vs With Family")

plt.tight_layout()
plt.show()

## 6. Prepare Data for Modeling

Select final features and split into train/test sets.

In [ ]:
features = ['Pclass', 'Sex_enc', 'Age', 'SibSp', 'Parch', 'Fare',
            'Embarked_enc', 'FamilySize', 'IsAlone', 'AgeGroup_enc']
X = df_fe[features]
y = df_fe['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## 7. Decision Tree Classifier

### 7.1 Train the Model

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=5, min_samples_split=10, random_state=42)
dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)

print("Decision Tree Accuracy:", round(accuracy_score(y_test, dt_pred), 4))
print("\nClassification Report:\n", classification_report(y_test, dt_pred))

### 7.2 Visualize the Decision Tree

In [ ]:
plt.figure(figsize=(20,10))
plot_tree(dt_model, feature_names=features, class_names=['Died','Survived'],
          filled=True, rounded=True, fontsize=8, max_depth=3)
plt.title("Decision Tree (top 3 levels shown)")
plt.show()

### 7.3 Decision Tree Confusion Matrix

In [ ]:
cm_dt = confusion_matrix(y_test, dt_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Died','Survived'], yticklabels=['Died','Survived'])
plt.title("Decision Tree - Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

## 8. Random Forest Classifier

### 8.1 Train the Model

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, max_depth=6,
                                    min_samples_split=10, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("\nClassification Report:\n", classification_report(y_test, rf_pred))

### 8.2 Feature Importance

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x=importances.values, y=importances.index, palette='crest')
plt.title("Random Forest - Feature Importance")
plt.xlabel("Importance")
plt.show()

importances

### 8.3 Random Forest Confusion Matrix

In [ ]:
cm_rf = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Died','Survived'], yticklabels=['Died','Survived'])
plt.title("Random Forest - Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

## 9. Model Comparison: Decision Tree vs Random Forest

In [ ]:
results = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest'],
    'Accuracy': [accuracy_score(y_test, dt_pred), accuracy_score(y_test, rf_pred)],
    'Precision': [precision_score(y_test, dt_pred), precision_score(y_test, rf_pred)],
    'Recall': [recall_score(y_test, dt_pred), recall_score(y_test, rf_pred)],
    'F1 Score': [f1_score(y_test, dt_pred), f1_score(y_test, rf_pred)]
})
results = results.round(4)
results

In [ ]:
results_melted = results.melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(9,5))
sns.barplot(x='Metric', y='Score', hue='Model', data=results_melted, palette='Set2')
plt.title("Decision Tree vs Random Forest - Metric Comparison")
plt.ylim(0,1)
plt.show()

### 9.1 ROC Curve Comparison

In [ ]:
dt_probs = dt_model.predict_proba(X_test)[:,1]
rf_probs = rf_model.predict_proba(X_test)[:,1]

fpr_dt, tpr_dt, _ = roc_curve(y_test, dt_probs)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probs)

plt.figure(figsize=(7,6))
plt.plot(fpr_dt, tpr_dt, label=f"Decision Tree (AUC = {auc(fpr_dt, tpr_dt):.3f})")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {auc(fpr_rf, tpr_rf):.3f})")
plt.plot([0,1],[0,1], linestyle='--', color='gray', label='Random Guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()

## 10. Conclusion

- We performed EDA on the Titanic dataset and found that **sex**, **passenger class**, and **fare** were strong indicators of survival — women and first-class passengers had notably higher survival rates.
- After feature engineering (handling missing values, creating `FamilySize`, `IsAlone`, and `AgeGroup`), we trained both a **Decision Tree** and a **Random Forest** classifier.
- The **Random Forest** model outperformed the single **Decision Tree** on accuracy, precision, recall, and F1-score, as expected since it aggregates many trees and reduces overfitting/variance.
- Feature importance from the Random Forest confirmed `Sex`, `Fare`, and `Pclass` as the most influential predictors of survival.

**Possible future improvements:** hyperparameter tuning with GridSearchCV, trying Gradient Boosting/XGBoost, and engineering title-based features from passenger names.